# Profiling API — Testes

Testa todos os endpoints de `/profiling`:
- `POST /profiling/ingest` — Ingestão + predição (CSV, background)
- `GET /profiling/vehicle/{veiculo_id}?target=` — Info de veículo

In [1]:
import httpx
import pandas as pd
from pathlib import Path

BASE_URL = "http://localhost:8010"
client = httpx.Client(base_url=BASE_URL, timeout=120)

In [2]:
TARGET = "h"
DATA_PATH = '../../../datasets/full/telemetria_movias2025_features.csv'

In [3]:

import sys
import os

sys.path.append(os.path.abspath('../..'))

from moviasai.data.utils import load_raw_data

import polars as pl

from pathlib import Path
from datetime import date, timedelta

from moviasai.data.utils import load_raw_data

df = load_raw_data(DATA_PATH)

out_dir = Path("../../../tmp/segments")
out_dir.mkdir(parents=True, exist_ok=True)

# Garantir coluna data como Date
df_seg = df.with_columns(pl.col("data").cast(pl.Date))

# Segmento 1: até 2025-08-24 (inclusive)
cutoff = date(2025, 8, 24)
seg1 = df_seg.filter(pl.col("data") <= cutoff)

segments = [seg1]

# Demais segmentos: a partir de 2025-08-25, em pedaços de 7 dias
start = date(2025, 8, 25)
max_date = df_seg["data"].max()

while start + timedelta(days=6) <= max_date:
    end = start + timedelta(days=6)
    seg = df_seg.filter((pl.col("data") >= start) & (pl.col("data") <= end))
    segments.append(seg)
    start = end + timedelta(days=1)

segment_paths = []
# Salvar
for i, seg in enumerate(segments, start=1):
    path = out_dir / f"segment_{i}.csv"
    seg.write_csv(path)
    segment_paths.append(path)
    print(f"segment_{i}.csv: {seg.shape[0]} linhas, {seg['data'].min()} a {seg['data'].max()}")

print(f"\nTotal: {len(segments)} segmentos")

segment_1.csv: 4759884 linhas, 2025-01-01 a 2025-08-24
segment_2.csv: 141183 linhas, 2025-08-25 a 2025-08-31
segment_3.csv: 141183 linhas, 2025-09-01 a 2025-09-07
segment_4.csv: 141183 linhas, 2025-09-08 a 2025-09-14
segment_5.csv: 141183 linhas, 2025-09-15 a 2025-09-21
segment_6.csv: 141183 linhas, 2025-09-22 a 2025-09-28
segment_7.csv: 141183 linhas, 2025-09-29 a 2025-10-05

Total: 7 segmentos


## 1. Ingestão + Predição via CSV

Envia CSV com colunas `veiculo_id, data, h_dia_clean, km_dia_clean`.
Executa ingestão de perfis e predição para ambos os targets (h, km) numa transação atómica.

In [4]:
def run_ingestion(segment_paths, idx=0):
    csv_path = segment_paths[idx]

    if csv_path.exists():
        with open(csv_path, "rb") as f:
            resp = client.post("/profiling/ingest", files={"file": (csv_path.name, f, "text/csv")})
        print(f"Status: {resp.status_code}")
    else:
        print(f"CSV não encontrado: {csv_path}")
    return resp

In [7]:
resp = run_ingestion(segment_paths, idx=6)
resp.json()

Status: 202


{'run_id': 7, 'message': 'Ingestão e predição submetidas em background.'}

In [9]:

# Listar todos os runs de ingestão
import pandas as pd

resp_all = client.get("/pipeline/runs", params={"step": "ingestion"})
print(f"Status: {resp_all.status_code}")
pd.DataFrame(resp_all.json())


Status: 200


,id,step,target,model_type,status,started_at,finished_at,error_message,metrics,artifacts,created_at
0,7,ingestion,global,None,completed,2026-04-27T00:51:15.263289,2026-04-27T00:53:42.985038,None,{'ingestion': {'daily_activity_inserted': 4595...,None,2026-04-27T00:51:15
1,6,ingestion,global,None,completed,2026-04-27T00:47:52.282871,2026-04-27T00:50:53.131214,None,{'ingestion': {'daily_activity_inserted': 4592...,None,2026-04-27T00:47:52
2,5,ingestion,global,None,completed,2026-04-27T00:44:39.721178,2026-04-27T00:47:15.717621,None,{'ingestion': {'daily_activity_inserted': 4621...,None,2026-04-27T00:44:39
3,4,ingestion,global,None,completed,2026-04-27T00:39:49.883467,2026-04-27T00:42:20.958032,None,{'ingestion': {'daily_activity_inserted': 4621...,None,2026-04-27T00:39:49
4,3,ingestion,global,None,completed,2026-04-27T00:37:10.616027,2026-04-27T00:39:38.874832,None,{'ingestion': {'daily_activity_inserted': 4580...,None,2026-04-27T00:37:10
5,2,ingestion,global,None,completed,2026-04-27T00:34:18.431674,2026-04-27T00:36:50.308909,None,{'ingestion': {'daily_activity_inserted': 4612...,None,2026-04-27T00:34:18
6,1,ingestion,global,None,completed,2026-04-27T00:31:12.747795,2026-04-27T00:34:01.265706,None,{'ingestion': {'daily_activity_inserted': 1303...,None,2026-04-27T00:31:12


## 2. Consultar informações de um veículo

Retorna perfil (features) e metadados para um veículo e target.

In [28]:
VEICULO_ID = 4
TARGET = "km"

resp = client.get(f"/profiling/vehicle/{VEICULO_ID}", params={"target": TARGET})
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'veiculo_id': 4,
 'target': 'km',
 'profile': {'cluster_0_km': 0.9999971389770508,
  'cluster_1_km': 2.9037378226348665e-06,
  'cluster_2_km': 2.4084597163853694e-11,
  'cycle_mean_fim_km': 12.709166666666674,
  'cycle_mean_inicio_km': 16.024999999999974,
  'cycle_mean_meio_km': 21.15277777777782,
  'cycle_prob_active_fim_km': 0.7,
  'cycle_prob_active_inicio_km': 0.7777777777777778,
  'cycle_prob_active_meio_km': 0.7222222222222222,
  'cycle_ratio_fim_inicio_km': 0.7930837233489357,
  'day_1_cv_km': 0.7132362995882223,
  'day_1_iqr_km': 11.400000000000093,
  'day_1_mean_km': 19.00208333333339,
  'day_1_p25_km': 13.400000000000093,
  'day_1_p75_km': 24.800000000000185,
  'day_1_prob_active_km': 0.9583333333333334,
  'day_1_std_km': 13.552975601133738,
  'day_2_cv_km': 0.9049523116568253,
  'day_2_iqr_km': 15.399999999999181,
  'day_2_mean_km': 20.241666666666536,
  'day_2_p25_km': 8.400000000000091,
  'day_2_p75_km': 23.799999999999272,
  'day_2_prob_active_km': 0.9166666666666666,
  

In [ ]:
# Testar com target 'h'
resp = client.get(f"/profiling/vehicle/{VEICULO_ID}", params={"target": "h"})
print(f"Status: {resp.status_code}")
resp.json()

In [ ]:
# Testar veículo inexistente → deve retornar 404
resp = client.get("/profiling/vehicle/999999", params={"target": "km"})
print(f"Status: {resp.status_code}")
resp.json()